# Data Preparation and Merging

This notebook focuses on preparing the full feature dataset for final modelling. 
The steps taken were:

- Transforming regional datasets from wide to long format using `pd.melt()` to create a consistent structure.
- Combining regional-level features such as HPI, property sales, and percentage change
- Merging national indicators like mortgage rates, bank base rates, and CPIH into a single DataFrame.
- Merging all of the above with the core datasets: median house prices and average house prices. 
- Engineering lag features on the target variable (`median_price`) in preparation for time series models.


This dataset will serve as the input for training future models.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.metrics import mean_squared_error

## Reshaping Regional Datasets from Wide to Long-Format

In [2]:
#Loading in data sets that are in the wide format
region_data = ['interim/propertysales.csv','interim/HPI.csv','interim/percentagechange.csv']
dfs = [pd.read_csv(f) for f in region_data]

In [3]:
dfs[0]

,Region code,Region,01/12/1995,01/03/1996,01/06/1996,01/09/1996,01/12/1996,01/03/1997,01/06/1997,01/09/1997,...,01/12/2021,01/03/2022,01/06/2022,01/09/2022,01/12/2022,01/03/2023,01/06/2023,01/09/2023,01/12/2023,01/03/2024
0,E92000001,England,762489,768428,793761,850069,923064,964295,1012020,1047336,...,1023932,936846,832813,818665,844474,802464,736742,667658,585680,541102
1,E12000001,North East,32822,32778,33595,35345,37933,38794,40117,41139,...,47294,45348,42994,41374,41761,40289,37944,34968,31047,28576
2,E12000002,North West,94347,94404,96662,101841,107608,111173,115380,119098,...,133114,124950,115751,110765,111774,106297,97787,89279,79145,73048
3,E12000003,Yorkshire and The Humber,71322,71362,72161,75746,80881,83743,87144,89666,...,98373,92136,85449,81823,83592,79798,73891,67976,60536,56116
4,E12000004,East Midlands,64596,64598,66636,71313,76761,79911,83752,86518,...,92760,86804,80139,76835,78726,74480,68025,61375,53642,49077
5,E12000005,West Midlands,71767,72261,74295,79098,85156,88607,92537,94532,...,96861,89864,82246,79074,80734,76629,71050,64373,56832,52599
6,E12000006,East of England,93136,93813,97403,104885,114757,120232,126859,132032,...,127708,114579,99573,99188,103146,97457,88763,78808,67698,62007
7,E12000007,London,108731,109784,113601,122597,135328,143337,151495,157776,...,114738,102978,84489,90829,97028,92922,84457,75960,66209,61371
8,E12000008,South East,142577,144316,150391,162163,177707,186291,196177,203278,...,190613,169546,145011,144973,150548,142246,129918,116717,101550,93923
9,E12000009,South West,83191,85112,89017,97081,106933,112207,118559,123297,...,122471,110641,97161,93804,97165,92346,84907,78202,69021,64385


In [4]:
#Empty list to hold data frames after reshaping
region_dfs = []

for df in dfs:
    #Identifying columns that would remain the same
    #Different to what was done to median price data as that method would remove dates
    id_vars = ['Region', 'Region code']
    value_vars = [col for col in df.columns if col not in id_vars]

    #Converting each date column into a row which will correspond to a region/code
    region_df = df.melt(
        id_vars=id_vars,
        value_vars=value_vars,
        var_name='Date',
        value_name='Value'
    )
    #Converting date into datetime object
    region_df['Date'] = pd.to_datetime(region_df['Date'], dayfirst=True, errors='coerce')

    #Adding reshaped data set to empty list
    region_dfs.append(region_df)
    

In [5]:
region_dfs[1]

,Region,Region code,Date,Value
0,England,E92000001,1995-01-01,17.6
1,North East,E12000001,1995-01-01,26.4
2,North West,E12000002,1995-01-01,20.7
3,Yorkshire and The Humber,E12000003,1995-01-01,22.0
4,East Midlands,E12000004,1995-01-01,18.6
...,...,...,...,...
3595,West Midlands,E12000005,2024-12-01,102.7
3596,East of England,E12000006,2024-12-01,99.9
3597,London,E12000007,2024-12-01,96.2
3598,South East,E12000008,2024-12-01,98.9


In [6]:
#Assigning names to each data frame
sales_df = region_dfs[0]
hpi_df = region_dfs[1]
pctchange_df = region_dfs[2]

#Renaming values column
sales_df.rename(columns={"Value": "prtySales"}, inplace=True)
hpi_df.rename(columns={"Value": "hpi"}, inplace=True)
pctchange_df.rename(columns={"Value": "perChange"}, inplace=True)

In [7]:
sales_df

,Region,Region code,Date,prtySales
0,England,E92000001,1995-12-01,762489
1,North East,E12000001,1995-12-01,32822
2,North West,E12000002,1995-12-01,94347
3,Yorkshire and The Humber,E12000003,1995-12-01,71322
4,East Midlands,E12000004,1995-12-01,64596
...,...,...,...,...
1135,West Midlands,E12000005,2024-03-01,52599
1136,East of England,E12000006,2024-03-01,62007
1137,London,E12000007,2024-03-01,61371
1138,South East,E12000008,2024-03-01,93923


## Merging Regional Datasets

In [8]:
#Merging all data frames into one
#Merged using outer in order to retain all data
regions_df = region_dfs[0].merge(region_dfs[1], on=['Region', 'Region code', 'Date'], how='outer')
regions_df = regions_df.merge(region_dfs[2], on=['Region', 'Region code', 'Date'], how='outer')


In [9]:
regions_df

,Region,Region code,Date,prtySales,hpi,perChange
0,East Midlands,E12000004,1995-01-01,NaN,18.6,-2.4
1,East Midlands,E12000004,1995-02-01,NaN,18.8,1.1
2,East Midlands,E12000004,1995-03-01,NaN,18.6,-1.4
3,East Midlands,E12000004,1995-04-01,NaN,18.9,1.6
4,East Midlands,E12000004,1995-05-01,NaN,18.8,-0.5
...,...,...,...,...,...,...
3605,Yorkshire and The Humber,E12000003,2024-09-01,NaN,105.6,-0.6
3606,Yorkshire and The Humber,E12000003,2024-10-01,NaN,106.0,0.7
3607,Yorkshire and The Humber,E12000003,2024-11-01,NaN,107.3,1.2
3608,Yorkshire and The Humber,E12000003,2024-12-01,NaN,106.4,-0.6


## Merging and Cleaning Nation Wide Datasets

In [10]:
#Loading in data sets that apply nationally
national_data = ['interim/bankBaseRate.csv','interim/Mortgage.csv','interim/UKCPIH.csv']

#Reading each dataset with date as index, ensuring it is read correctly
dfs = [pd.read_csv(f, parse_dates=True, index_col=0, dayfirst=True) for f in national_data]

#No regions - able to concat using date as the index
national_data = pd.concat(dfs,axis=1)

In [11]:
national_data

,Rate,5 Year fixed (75% LTV),2 year fixed 75% LTV,Revert to rate,UKCPIH
Date,,,,,
1994-12-07,6.13,NaN,NaN,NaN,NaN
1995-01-01,NaN,NaN,NaN,NaN,2.4
1995-01-31,NaN,9.44,8.13,8.21,NaN
1995-02-01,NaN,NaN,NaN,NaN,2.3
1995-02-02,6.63,NaN,NaN,NaN,NaN
...,...,...,...,...,...
2024-12-01,NaN,NaN,NaN,NaN,3.5
2024-12-31,NaN,4.38,4.64,7.46,NaN
2025-01-01,NaN,NaN,NaN,NaN,3.9


In [12]:
#Removing date as index
national_data = national_data.reset_index()

#Converting date column to datetime object and setting it as index
national_data['Date'] = pd.to_datetime(national_data['Date'])
national_data = national_data.set_index('Date')

#Using forward fill to fill missing values
national_data = national_data.ffill()

#Filtering rows where the date is the first of the month
national_data = national_data[national_data.index.day == 1]

In [13]:
#Removing date as index in preparation for merging
national_data = national_data.reset_index()
national_data

,Date,Rate,5 Year fixed (75% LTV),2 year fixed 75% LTV,Revert to rate,UKCPIH
0,1995-01-01,6.13,NaN,NaN,NaN,2.4
1,1995-02-01,6.13,9.44,8.13,8.21,2.3
2,1995-03-01,6.63,9.38,8.38,8.33,2.4
3,1995-04-01,6.63,9.33,8.08,8.36,2.3
4,1995-05-01,6.63,9.40,8.29,8.36,2.6
...,...,...,...,...,...,...
356,2024-09-01,5.00,4.33,4.58,7.70,2.6
357,2024-10-01,5.00,4.09,4.40,7.68,3.2
358,2024-11-01,5.00,4.07,4.53,7.56,3.5
359,2024-12-01,4.75,4.31,4.60,7.47,3.5


# Merging All Datasets

In [14]:
#Loading in price related data sets
#Already in Long format, no need to reshape
price_data = ['interim/monthly_median_prices.csv','interim/longAverage.csv']
price_dfs = [pd.read_csv(f) for f in price_data]

In [15]:
price_dfs[0]

,Local authority code,Date,Region code,Region,LocalAuthority,Price
0,eng,1995-12-01,E92000001,England,England,55000.00
1,eng,1996-01-01,E92000001,England,England,55000.00
2,eng,1996-02-01,E92000001,England,England,55000.00
3,eng,1996-03-01,E92000001,England,England,55000.00
4,eng,1996-04-01,E92000001,England,England,55166.67
...,...,...,...,...,...,...
100975,E07000083,2023-11-01,E12000009,South West,Tewkesbury,352141.00
100976,E07000083,2023-12-01,E12000009,South West,Tewkesbury,352199.50
100977,E07000083,2024-01-01,E12000009,South West,Tewkesbury,350149.67
100978,E07000083,2024-02-01,E12000009,South West,Tewkesbury,348099.83


In [16]:
#Assiging names to data frames
median_df = price_dfs[0]
average_df = price_dfs[1]

#Renaming columns
median_df.rename(columns={'Price': 'median_price'}, inplace=True)
average_df.rename(columns={'Price': 'average_price'}, inplace=True)

In [17]:
median_df

,Local authority code,Date,Region code,Region,LocalAuthority,median_price
0,eng,1995-12-01,E92000001,England,England,55000.00
1,eng,1996-01-01,E92000001,England,England,55000.00
2,eng,1996-02-01,E92000001,England,England,55000.00
3,eng,1996-03-01,E92000001,England,England,55000.00
4,eng,1996-04-01,E92000001,England,England,55166.67
...,...,...,...,...,...,...
100975,E07000083,2023-11-01,E12000009,South West,Tewkesbury,352141.00
100976,E07000083,2023-12-01,E12000009,South West,Tewkesbury,352199.50
100977,E07000083,2024-01-01,E12000009,South West,Tewkesbury,350149.67
100978,E07000083,2024-02-01,E12000009,South West,Tewkesbury,348099.83


In [18]:
#Converting date column to datetime object and ensuring correct date format
median_df['Date'] = pd.to_datetime(median_df['Date'], format="%Y-%m-%d")
average_df['Date'] = pd.to_datetime(average_df['Date'], format="%Y-%m-%d")

In [19]:
#Merging median with average house prices using Region, Region code, and Date
full_data = median_df.merge(average_df, on=['Region','Date','Region code'], how='left')

In [20]:
full_data

,Local authority code,Date,Region code,Region,LocalAuthority,median_price,average_price
0,eng,1995-12-01,E92000001,England,England,55000.00,49965
1,eng,1996-01-01,E92000001,England,England,55000.00,49410
2,eng,1996-02-01,E92000001,England,England,55000.00,49601
3,eng,1996-03-01,E92000001,England,England,55000.00,49739
4,eng,1996-04-01,E92000001,England,England,55166.67,50352
...,...,...,...,...,...,...,...
100975,E07000083,2023-11-01,E12000009,South West,Tewkesbury,352141.00,300000
100976,E07000083,2023-12-01,E12000009,South West,Tewkesbury,352199.50,295000
100977,E07000083,2024-01-01,E12000009,South West,Tewkesbury,350149.67,299000
100978,E07000083,2024-02-01,E12000009,South West,Tewkesbury,348099.83,298000


In [21]:
#Creating region level map
region_code_map = {
    'North East': 'E12000001',
    'North West': 'E12000002',
    'Yorkshire and The Humber': 'E12000003',
    'East Midlands': 'E12000004',
    'West Midlands': 'E12000005',
    'East of England': 'E12000006',
    'London': 'E12000007',
    'South East': 'E12000008',
    'South West': 'E12000009',
    'England': 'E92000001'
}


In [ ]:
Here I am creating rows for regional data as their data is by local authority. This will allow user to be able t

In [22]:
#Filter out England data
region_level_data = full_data[full_data['Region'] != 'England'].copy()

#Group regional data
region_summary = (
    region_level_data
    .groupby(['Date', 'Region'])[['median_price', 'average_price']]
    .mean()
    .reset_index()
)

#Add placeholders
region_summary['LocalAuthority'] = 'ALL'
region_summary['Local authority code'] = 'ALL'
region_summary['Region code'] = region_summary['Region'].map(region_code_map)

#Reorder
cols = full_data.columns.tolist()
region_summary = region_summary[cols]

#Appending regional rows to full_data
combined_data = pd.concat([full_data, region_summary], ignore_index=True)



In [23]:
combined_data


,Local authority code,Date,Region code,Region,LocalAuthority,median_price,average_price
0,eng,1995-12-01,E92000001,England,England,55000.000000,49965.0
1,eng,1996-01-01,E92000001,England,England,55000.000000,49410.0
2,eng,1996-02-01,E92000001,England,England,55000.000000,49601.0
3,eng,1996-03-01,E92000001,England,England,55000.000000,49739.0
4,eng,1996-04-01,E92000001,England,England,55166.670000,50352.0
...,...,...,...,...,...,...,...
104035,ALL,2024-03-01,E12000002,North West,ALL,205930.169143,198000.0
104036,ALL,2024-03-01,E12000008,South East,ALL,414990.840313,368000.0
104037,ALL,2024-03-01,E12000009,South West,ALL,330488.680370,296000.0
104038,ALL,2024-03-01,E12000005,West Midlands,ALL,258296.851333,232000.0


In [24]:
#Merging regional data to main dataset
full_data = combined_data.merge(regions_df, on=['Region','Date','Region code'], how='left')

In [25]:
full_data

,Local authority code,Date,Region code,Region,LocalAuthority,median_price,average_price,prtySales,hpi,perChange
0,eng,1995-12-01,E92000001,England,England,55000.000000,49965.0,762489.0,17.5,0.3
1,eng,1996-01-01,E92000001,England,England,55000.000000,49410.0,NaN,17.3,-1.1
2,eng,1996-02-01,E92000001,England,England,55000.000000,49601.0,NaN,17.4,0.4
3,eng,1996-03-01,E92000001,England,England,55000.000000,49739.0,768428.0,17.4,0.3
4,eng,1996-04-01,E92000001,England,England,55166.670000,50352.0,NaN,17.6,1.2
...,...,...,...,...,...,...,...,...,...,...
104035,ALL,2024-03-01,E12000002,North West,ALL,205930.169143,198000.0,73048.0,100.5,1.2
104036,ALL,2024-03-01,E12000008,South East,ALL,414990.840313,368000.0,93923.0,94.8,-0.4
104037,ALL,2024-03-01,E12000009,South West,ALL,330488.680370,296000.0,64385.0,95.8,-0.7
104038,ALL,2024-03-01,E12000005,West Midlands,ALL,258296.851333,232000.0,52599.0,97.6,-0.3


In [26]:
#Merging national data to main dataset
full_data = full_data.merge(national_data, on=['Date'], how='left')

In [27]:
full_data

,Local authority code,Date,Region code,Region,LocalAuthority,median_price,average_price,prtySales,hpi,perChange,Rate,5 Year fixed (75% LTV),2 year fixed 75% LTV,Revert to rate,UKCPIH
0,eng,1995-12-01,E92000001,England,England,55000.000000,49965.0,762489.0,17.5,0.3,6.63,8.14,7.99,7.96,3.1
1,eng,1996-01-01,E92000001,England,England,55000.000000,49410.0,NaN,17.3,-1.1,6.38,8.13,7.99,7.95,2.9
2,eng,1996-02-01,E92000001,England,England,55000.000000,49601.0,NaN,17.4,0.4,6.13,7.87,6.70,7.72,3.0
3,eng,1996-03-01,E92000001,England,England,55000.000000,49739.0,768428.0,17.4,0.3,6.13,7.66,6.65,7.50,3.0
4,eng,1996-04-01,E92000001,England,England,55166.670000,50352.0,NaN,17.6,1.2,5.94,7.70,6.49,7.46,2.8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
104035,ALL,2024-03-01,E12000002,North West,ALL,205930.169143,198000.0,73048.0,100.5,1.2,5.25,4.41,4.96,7.93,3.8
104036,ALL,2024-03-01,E12000008,South East,ALL,414990.840313,368000.0,93923.0,94.8,-0.4,5.25,4.41,4.96,7.93,3.8
104037,ALL,2024-03-01,E12000009,South West,ALL,330488.680370,296000.0,64385.0,95.8,-0.7,5.25,4.41,4.96,7.93,3.8
104038,ALL,2024-03-01,E12000005,West Midlands,ALL,258296.851333,232000.0,52599.0,97.6,-0.3,5.25,4.41,4.96,7.93,3.8


## Cleaning Up Final Dataset

In [28]:
#Rearranging column order
new_order = [
    'Date', 'Region', 'Region code', 'Local authority code', 'LocalAuthority',
    'median_price', 'average_price', 'prtySales','hpi','perChange','Rate','5 Year fixed (75% LTV)','2 year fixed 75% LTV','Revert to rate','UKCPIH'
]

#adding columns in order
remaining = [col for col in full_data.columns if col not in new_order]
new_order += remaining

# Reorder the DataFrame
full_data = full_data[new_order]

In [29]:
full_data

,Date,Region,Region code,Local authority code,LocalAuthority,median_price,average_price,prtySales,hpi,perChange,Rate,5 Year fixed (75% LTV),2 year fixed 75% LTV,Revert to rate,UKCPIH
0,1995-12-01,England,E92000001,eng,England,55000.000000,49965.0,762489.0,17.5,0.3,6.63,8.14,7.99,7.96,3.1
1,1996-01-01,England,E92000001,eng,England,55000.000000,49410.0,NaN,17.3,-1.1,6.38,8.13,7.99,7.95,2.9
2,1996-02-01,England,E92000001,eng,England,55000.000000,49601.0,NaN,17.4,0.4,6.13,7.87,6.70,7.72,3.0
3,1996-03-01,England,E92000001,eng,England,55000.000000,49739.0,768428.0,17.4,0.3,6.13,7.66,6.65,7.50,3.0
4,1996-04-01,England,E92000001,eng,England,55166.670000,50352.0,NaN,17.6,1.2,5.94,7.70,6.49,7.46,2.8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
104035,2024-03-01,North West,E12000002,ALL,ALL,205930.169143,198000.0,73048.0,100.5,1.2,5.25,4.41,4.96,7.93,3.8
104036,2024-03-01,South East,E12000008,ALL,ALL,414990.840313,368000.0,93923.0,94.8,-0.4,5.25,4.41,4.96,7.93,3.8
104037,2024-03-01,South West,E12000009,ALL,ALL,330488.680370,296000.0,64385.0,95.8,-0.7,5.25,4.41,4.96,7.93,3.8
104038,2024-03-01,West Midlands,E12000005,ALL,ALL,258296.851333,232000.0,52599.0,97.6,-0.3,5.25,4.41,4.96,7.93,3.8


In [30]:
full_data = full_data.set_index('Date')

In [31]:
full_data

,Region,Region code,Local authority code,LocalAuthority,median_price,average_price,prtySales,hpi,perChange,Rate,5 Year fixed (75% LTV),2 year fixed 75% LTV,Revert to rate,UKCPIH
Date,,,,,,,,,,,,,,
1995-12-01,England,E92000001,eng,England,55000.000000,49965.0,762489.0,17.5,0.3,6.63,8.14,7.99,7.96,3.1
1996-01-01,England,E92000001,eng,England,55000.000000,49410.0,NaN,17.3,-1.1,6.38,8.13,7.99,7.95,2.9
1996-02-01,England,E92000001,eng,England,55000.000000,49601.0,NaN,17.4,0.4,6.13,7.87,6.70,7.72,3.0
1996-03-01,England,E92000001,eng,England,55000.000000,49739.0,768428.0,17.4,0.3,6.13,7.66,6.65,7.50,3.0
1996-04-01,England,E92000001,eng,England,55166.670000,50352.0,NaN,17.6,1.2,5.94,7.70,6.49,7.46,2.8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-03-01,North West,E12000002,ALL,ALL,205930.169143,198000.0,73048.0,100.5,1.2,5.25,4.41,4.96,7.93,3.8
2024-03-01,South East,E12000008,ALL,ALL,414990.840313,368000.0,93923.0,94.8,-0.4,5.25,4.41,4.96,7.93,3.8
2024-03-01,South West,E12000009,ALL,ALL,330488.680370,296000.0,64385.0,95.8,-0.7,5.25,4.41,4.96,7.93,3.8


In [32]:
#Renaming columns for convenience
full_data.columns = ['region','regionCode','LAcode','LA','median','average','sales','hpi','change','base','5year','2year','revert','cpih']

In [33]:
full_data

,region,regionCode,LAcode,LA,median,average,sales,hpi,change,base,5year,2year,revert,cpih
Date,,,,,,,,,,,,,,
1995-12-01,England,E92000001,eng,England,55000.000000,49965.0,762489.0,17.5,0.3,6.63,8.14,7.99,7.96,3.1
1996-01-01,England,E92000001,eng,England,55000.000000,49410.0,NaN,17.3,-1.1,6.38,8.13,7.99,7.95,2.9
1996-02-01,England,E92000001,eng,England,55000.000000,49601.0,NaN,17.4,0.4,6.13,7.87,6.70,7.72,3.0
1996-03-01,England,E92000001,eng,England,55000.000000,49739.0,768428.0,17.4,0.3,6.13,7.66,6.65,7.50,3.0
1996-04-01,England,E92000001,eng,England,55166.670000,50352.0,NaN,17.6,1.2,5.94,7.70,6.49,7.46,2.8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-03-01,North West,E12000002,ALL,ALL,205930.169143,198000.0,73048.0,100.5,1.2,5.25,4.41,4.96,7.93,3.8
2024-03-01,South East,E12000008,ALL,ALL,414990.840313,368000.0,93923.0,94.8,-0.4,5.25,4.41,4.96,7.93,3.8
2024-03-01,South West,E12000009,ALL,ALL,330488.680370,296000.0,64385.0,95.8,-0.7,5.25,4.41,4.96,7.93,3.8


In [34]:
full_data = full_data.sort_values(['region','LA', 'Date']).reset_index()
full_data

,Date,region,regionCode,LAcode,LA,median,average,sales,hpi,change,base,5year,2year,revert,cpih
0,1995-12-01,East Midlands,E12000004,ALL,ALL,48882.446000,43957.0,64596.0,18.7,1.5,6.63,8.14,7.99,7.96,3.1
1,1996-01-01,East Midlands,E12000004,ALL,ALL,48915.630857,42965.0,NaN,18.3,-2.3,6.38,8.13,7.99,7.95,2.9
2,1996-02-01,East Midlands,E12000004,ALL,ALL,48948.815714,42856.0,NaN,18.2,-0.3,6.13,7.87,6.70,7.72,3.0
3,1996-03-01,East Midlands,E12000004,ALL,ALL,48982.000571,43515.0,64598.0,18.5,1.5,6.13,7.66,6.65,7.50,3.0
4,1996-04-01,East Midlands,E12000004,ALL,ALL,49046.851143,44010.0,NaN,18.7,1.1,5.94,7.70,6.49,7.46,2.8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
104035,2023-11-01,Yorkshire and The Humber,E12000003,E06000014,York,315085.760000,194000.0,NaN,101.2,-0.7,5.25,5.02,5.28,8.00,4.2
104036,2023-12-01,Yorkshire and The Humber,E12000003,E06000014,York,316014.580000,193000.0,60536.0,100.5,-0.7,5.25,4.89,5.03,7.92,4.2
104037,2024-01-01,Yorkshire and The Humber,E12000003,E06000014,York,317287.120000,192000.0,NaN,100.0,-0.5,5.25,4.68,4.71,7.94,4.2
104038,2024-02-01,Yorkshire and The Humber,E12000003,E06000014,York,318559.650000,191000.0,NaN,99.9,-0.1,5.25,4.43,4.76,7.94,3.8


In [35]:
full_data = full_data.set_index('Date')


In [36]:
full_data['regionCode'] = full_data['regionCode'].str.replace('E', '', regex=False)
full_data['LAcode'] = full_data['LAcode'].str.replace('E', '', regex=False)

In [37]:
full_data['regionCode'] = pd.to_numeric(full_data['regionCode'], errors='coerce').astype('Int64')

# Define map for region-level LAcodes
region_la_map = {
    12000001: 1,
    12000002: 2,
    12000003: 3,
    12000004: 4,
    12000005: 5,
    12000006: 6,
    12000007: 7,
    12000008: 8,
    12000009: 9,
    92000001: 10  # England-level, optional
}

# Assign numeric LAcode for region-level rows
region_mask = full_data['LAcode'].isin(['ALL', 'eng'])
full_data.loc[region_mask, 'LAcode'] = full_data.loc[region_mask, 'regionCode'].map(region_la_map)

full_data['LAcode'] = pd.to_numeric(full_data['LAcode'], errors='coerce').astype('Int64')



In [38]:
full_data

,region,regionCode,LAcode,LA,median,average,sales,hpi,change,base,5year,2year,revert,cpih
Date,,,,,,,,,,,,,,
1995-12-01,East Midlands,12000004,4,ALL,48882.446000,43957.0,64596.0,18.7,1.5,6.63,8.14,7.99,7.96,3.1
1996-01-01,East Midlands,12000004,4,ALL,48915.630857,42965.0,NaN,18.3,-2.3,6.38,8.13,7.99,7.95,2.9
1996-02-01,East Midlands,12000004,4,ALL,48948.815714,42856.0,NaN,18.2,-0.3,6.13,7.87,6.70,7.72,3.0
1996-03-01,East Midlands,12000004,4,ALL,48982.000571,43515.0,64598.0,18.5,1.5,6.13,7.66,6.65,7.50,3.0
1996-04-01,East Midlands,12000004,4,ALL,49046.851143,44010.0,NaN,18.7,1.1,5.94,7.70,6.49,7.46,2.8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2023-11-01,Yorkshire and The Humber,12000003,6000014,York,315085.760000,194000.0,NaN,101.2,-0.7,5.25,5.02,5.28,8.00,4.2
2023-12-01,Yorkshire and The Humber,12000003,6000014,York,316014.580000,193000.0,60536.0,100.5,-0.7,5.25,4.89,5.03,7.92,4.2
2024-01-01,Yorkshire and The Humber,12000003,6000014,York,317287.120000,192000.0,NaN,100.0,-0.5,5.25,4.68,4.71,7.94,4.2


## Creating Lag Features

In [39]:
for lag in range(1, 13):
    full_data[f'lag_{lag}'] = (
        full_data.groupby(['region', 'LA'])['median'].shift(lag)
    )


In [40]:
full_data

,region,regionCode,LAcode,LA,median,average,sales,hpi,change,base,...,lag_3,lag_4,lag_5,lag_6,lag_7,lag_8,lag_9,lag_10,lag_11,lag_12
Date,,,,,,,,,,,,,,,,,,,,,
1995-12-01,East Midlands,12000004,4,ALL,48882.446000,43957.0,64596.0,18.7,1.5,6.63,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1996-01-01,East Midlands,12000004,4,ALL,48915.630857,42965.0,NaN,18.3,-2.3,6.38,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1996-02-01,East Midlands,12000004,4,ALL,48948.815714,42856.0,NaN,18.2,-0.3,6.13,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1996-03-01,East Midlands,12000004,4,ALL,48982.000571,43515.0,64598.0,18.5,1.5,6.13,...,48882.446000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1996-04-01,East Midlands,12000004,4,ALL,49046.851143,44010.0,NaN,18.7,1.1,5.94,...,48915.630857,48882.446,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2023-11-01,Yorkshire and The Humber,12000003,6000014,York,315085.760000,194000.0,NaN,101.2,-0.7,5.25,...,312500.210000,311772.300,311044.38,310768.91,310493.43,310217.96,309263.02,308308.07,307353.13,305956.00
2023-12-01,Yorkshire and The Humber,12000003,6000014,York,316014.580000,193000.0,60536.0,100.5,-0.7,5.25,...,313228.130000,312500.210,311772.30,311044.38,310768.91,310493.43,310217.96,309263.02,308308.07,307353.13
2024-01-01,Yorkshire and The Humber,12000003,6000014,York,317287.120000,192000.0,NaN,100.0,-0.5,5.25,...,314156.950000,313228.130,312500.21,311772.30,311044.38,310768.91,310493.43,310217.96,309263.02,308308.07


In [45]:
# Assuming your DataFrame is named full_data
unique_las = full_data[full_data['LA'] != 'ALL'][['LAcode', 'LA']].drop_duplicates().sort_values('LA')

# Optional: Reset index for cleaner output
unique_las = unique_las.reset_index(drop=True)

# View the result
print(unique_las.head())

# Save to CSV if needed
unique_las.to_csv('processed/unique_la_codes.csv', index=False)


    LAcode            LA
0  7000223          Adur
1  7000032  Amber Valley
2  7000224          Arun
3  7000170      Ashfield
4  7000105       Ashford


In [41]:
def create_features(df):
    df = df.copy() 
    df['quarter'] = df.index.quarter
    df['month'] = df.index.month
    df['year'] = df.index.year
    return df
    
full_data = create_features(full_data)


In [42]:
print(full_data.isna().sum())

region            0
regionCode        0
LAcode            0
LA                0
median            0
average           0
sales         69156
hpi               0
change            0
base              0
5year             0
2year             0
revert            0
cpih              0
lag_1           306
lag_2           612
lag_3           918
lag_4          1224
lag_5          1530
lag_6          1836
lag_7          2142
lag_8          2448
lag_9          2754
lag_10         3060
lag_11         3366
lag_12         3672
quarter           0
month             0
year              0
dtype: int64


Here I have identified there are many missing values for the property sales, this is due to the fact that the data I have collected is quarterly compared to the other monthly based data I have collected. As this data is time sensitive I have opted against using forward fill to reduce the number of missing values.

As well as this, looking as the relationship between median house prices and the number of property sales, it is likely that the changes in house prices influences the volume of property sales rather than the number of sales determining house prices. As median price will be my target variable, including such a feature could introduce the risk of adding noise into my model. For this reason I have decided to remove property sales as a feature of my model.

In [43]:
#Removing property sales data
full_data = full_data.drop(columns=['sales'])

## Exporting Complete Dataset

In [44]:
full_data.to_csv('processed/model_data.csv', index=True)